# Convex QP & LP with `pounce.qp`

POUNCE ships a specialized **convex conic interior-point solver**
(`pounce-convex`) alongside the general NLP filter-IPM. This notebook is the
gentle, build-up introduction to its Python surface, `pounce.qp`, for
linear and quadratic programs:

$$
\min_x\;\tfrac12 x^\top P x + c^\top x
\quad\text{s.t.}\quad
A x = b,\;\; G x \le h,\;\; \text{lb} \le x \le \text{ub}.
$$

`P = 0` is an LP; `P \succeq 0` a convex QP. We start with a one-line LP and
work up to **duals**, **verified infeasibility**, **warm starting**,
**parallel batches**, and **factorization reuse**.

> The conic interior-point design follows
> [Clarabel](https://github.com/oxfordcontrol/Clarabel.rs) (Goulart & Chen)
> and the presolve follows [PaPILO](https://github.com/scipopt/papilo) (the
> presolving library of [SCIP](https://www.scipopt.org/)). POUNCE is pure
> Rust and wraps neither — it ports their ideas.

In [1]:
import numpy as np
from pounce.qp import solve_qp, solve_socp, solve_qp_batch, QpFactorization

np.set_printoptions(precision=4, suppress=True)

## 1. The simplest LP

Minimize $-x_0 - x_1$ over the box $0 \le x \le 1$ subject to
$x_0 + x_1 \le 1$. The optimum sits on the constraint: any point with
$x_0 + x_1 = 1$ ties at objective $-1$.

In [2]:
r = solve_qp(
    c=[-1.0, -1.0],            # P=None -> linear objective
    G=[[1.0, 1.0]], h=[1.0],   # x0 + x1 <= 1
    lb=[0, 0], ub=[1, 1],
)
print(r.status, "  x =", r.x, "  obj =", r.obj, "  iters =", r.iters)
assert r.success and abs(r.obj + 1.0) < 1e-6

optimal   x = [0.5 0.5]   obj = -0.9999999994887762   iters = 7


## 2. A quadratic objective, with duals

$$\min_x\; \tfrac12\cdot 2\|x\|^2 - 3x_0 - 4x_1
\quad\text{s.t.}\quad x_0 + x_1 \le 1,\; 0 \le x \le 1.$$

The unconstrained minimizer of $\tfrac12\cdot2\|x\|^2-3x_0-4x_1$ is
$(1.5, 2)$, which violates $x_0+x_1\le1$, so the inequality is **active**.
The result carries the full multiplier set.

In [3]:
r = solve_qp(
    P=np.diag([2.0, 2.0]),
    c=[-3.0, -4.0],
    G=[[1.0, 1.0]], h=[1.0],
    lb=[0, 0], ub=[1, 1],
)
print("status :", r.status)
print("x      :", r.x)
print("obj    :", r.obj)
print("z (ineq):", r.z, "   <- > 0 means x0+x1<=1 is active")
print("z_lb    :", r.z_lb)
print("z_ub    :", r.z_ub)
assert r.success and abs(r.x.sum() - 1.0) < 1e-6

status : optimal
x      : [0.25 0.75]
obj    : -3.124999999715122
z (ineq): [2.5]    <- > 0 means x0+x1<=1 is active
z_lb    : [0. 0.]
z_ub    : [0. 0.]


### Stationarity check (KKT)

At the optimum the gradient of the Lagrangian vanishes:
$$Px + c + G^\top z - z_{lb} + z_{ub} = 0.$$
We verify the multipliers POUNCE returns actually close the KKT system.

In [4]:
P = np.diag([2.0, 2.0]); c = np.array([-3.0, -4.0]); G = np.array([[1.0, 1.0]])
stat = P @ r.x + c + G.T @ r.z - r.z_lb + r.z_ub
print("Lagrangian gradient:", stat, "  (~0)")
assert np.linalg.norm(stat) < 1e-6

Lagrangian gradient: [-0. -0.]   (~0)


## 3. Equality constraints

Project the origin's shifted point onto an affine subspace:
$$\min_x \tfrac12\|x\|^2 - x^\top p \quad\text{s.t.}\quad \mathbf 1^\top x = 1.$$
The closed-form solution is $x = p + \lambda\mathbf 1$ with $\lambda$ set so
the sum is 1, i.e. $x_i = p_i + (1-\sum p)/n$.

In [5]:
p = np.array([0.2, 0.5, 0.1])
n = p.size
r = solve_qp(P=np.eye(n), c=-p, A=np.ones((1, n)), b=[1.0])
x_star = p + (1 - p.sum()) / n
print("x        :", r.x)
print("closed   :", x_star)
print("y (eq)   :", r.y)
assert np.allclose(r.x, x_star, atol=1e-7)

x        : [0.2667 0.5667 0.1667]
closed   : [0.2667 0.5667 0.1667]
y (eq)   : [-0.0667]


## 4. Verified infeasibility & unboundedness

POUNCE reports **certified** status, not an iteration-limit guess: a Farkas
certificate for primal infeasibility, a recession ray for unboundedness.

In [6]:
# Infeasible: x >= 2 (via -x <= -2) AND x <= 1.
bad = solve_qp(c=[1.0], G=[[-1.0]], h=[-2.0], ub=[1.0])
print("infeasible case :", bad.status)

# Unbounded LP: minimize -x with no upper bound.
unb = solve_qp(c=[-1.0], lb=[0.0])
print("unbounded case  :", unb.status)

infeasible case : primal_infeasible
unbounded case  : dual_infeasible


## 5. Warm starting

Feed a previous (or nearby) solution back to seed the interior-point
iteration — the payoff for **parametric sweeps**, receding-horizon MPC, and
branch-and-bound subproblems. The warm start changes only the iteration
count, never the solution.

We sweep the linear term `c` along a path and reuse each solution to seed
the next.

In [7]:
P = np.diag([2.0, 2.0])
G = np.array([[1.0, 1.0]]); h = [1.0]
lb, ub = [0, 0], [1, 1]

cold_iters, warm_iters = [], []
prev = None
for t in np.linspace(0, 1, 12):
    c = [-3.0 - t, -4.0 + 2 * t]
    cold = solve_qp(P=P, c=c, G=G, h=h, lb=lb, ub=ub)
    warm = solve_qp(P=P, c=c, G=G, h=h, lb=lb, ub=ub, warm_start=prev)
    assert np.allclose(cold.x, warm.x, atol=1e-3)   # same solution (to tol; last point sits at a corner)
    cold_iters.append(cold.iters)
    warm_iters.append(warm.iters)
    prev = warm

print("cold iters:", cold_iters)
print("warm iters:", warm_iters)
print(f"mean cold = {np.mean(cold_iters):.1f}, mean warm = {np.mean(warm_iters[1:]):.1f}")

cold iters: [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 12]
warm iters: [8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 8]
mean cold = 8.4, mean warm = 6.3


## 6. Parallel batches

`solve_qp_batch` solves many independent QPs across a rayon thread pool
(outer-parallel across instances, serial within each). Pass a list of
kwarg dicts — each is exactly a `solve_qp` call.

In [8]:
rng = np.random.default_rng(0)
cs = [(-3.0 + 0.5 * k, -4.0 - 0.3 * k) for k in range(8)]
problems = [dict(P=P, c=c, G=G, h=h, lb=lb, ub=ub) for c in cs]
results = solve_qp_batch(problems)
print("all optimal:", all(r.success for r in results))
for c, r in zip(cs[:3], results[:3]):
    print(f"  c={np.array(c)}  ->  x={r.x}")

all optimal: True
  c=[-3. -4.]  ->  x=[0.25 0.75]
  c=[-2.5 -4.3]  ->  x=[0.05 0.95]
  c=[-2.  -4.6]  ->  x=[0. 1.]


You can also chain batches with `warm_starts=` — one warm start per
problem — to combine batching with warm starting across a sequence of
nearby batches.

In [9]:
nxt = solve_qp_batch(problems, warm_starts=results)
print("warm batch all optimal:", all(r.success for r in nxt))
print("solutions unchanged   :", all(np.allclose(a.x, b.x, atol=1e-7)
                                      for a, b in zip(results, nxt)))

warm batch all optimal: True
solutions unchanged   : True


## 7. Factorization reuse (build-once / solve-many)

When only the *values* of `c`/`b`/`h`/bounds change but the **structure**
(sparsity, the set of finite bounds) is fixed, `QpFactorization` builds the
AMD ordering and symbolic factor **once** and reuses it for every solve.
Compose it with warm starting for the fastest parametric loop.

In [10]:
fac = QpFactorization(P=P, c=[-3.0, -4.0], G=G, h=h, lb=lb, ub=ub)
prev = None
for t in np.linspace(0, 1, 5):
    c = [-3.0 - t, -4.0 + 2 * t]
    rk = fac.solve(P=P, c=c, G=G, h=h, lb=lb, ub=ub, warm_start=prev)
    print(f"t={t:.2f}  x={rk.x}  iters={rk.iters}")
    prev = rk

t=0.00  x=[0.25 0.75]  iters=8
t=0.25  x=[0.4375 0.5625]  iters=7
t=0.50  x=[0.625 0.375]  iters=7
t=0.75  x=[0.8125 0.1875]  iters=7
t=1.00  x=[0.9999 0.0001]  iters=9


## Where next

- **`14_socp.ipynb`** — second-order cone programs (norm minimization,
  robust LP, mixed cones) with the same API plus a `cones=` partition.
- **`15_differentiable_convex.ipynb`** — `pounce.jax`: differentiate the QP
  and SOCP solutions w.r.t. their data with `jax.grad` / `jacrev` / `vmap`.
- The [Convex Solver chapter](../../docs/src/convex-solver.md) documents the
  full API, presolve, and the differentiable layers.